In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
os.chdir("/content/drive/MyDrive/Smiles_data")
os.getcwd()

'/content/drive/MyDrive/Smiles_data'

In [ ]:
!pip install rdkit


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.3/34.3 MB 63.3 MB/s eta 0:00:00


In [ ]:
pip install tensorflow


In [ ]:
!pip install selfies


In [ ]:
import selfies as sf


**Preprocessing data**

In [ ]:
import pandas as pd
from rdkit import Chem
from tqdm import tqdm  # For progress tracking

# Load and clean SMILES data
def load_and_clean_data(file_path):
    # Load the CSV file
    df = pd.read_csv("/content/drive/MyDrive/Smiles_data/250k_rndm_zinc_drugs_clean_3.csv")

    # Ensure column name is lowercase
    df.columns = df.columns.str.lower()

    # Check if "smiles" column exists
    if "smiles" not in df.columns:
        raise ValueError("SMILES column not found. Check column names: " + str(df.columns))

    # Remove NaN values and duplicates
    smiles_list = df["smiles"].dropna().unique()

    # Validate SMILES
    valid_smiles = []
    for smi in tqdm(smiles_list, desc="Validating SMILES"):
        try:
            if Chem.MolFromSmiles(smi) is not None:  # Valid SMILES check
                valid_smiles.append(smi)
        except:
            continue  # Skip invalid SMILES

    # Save cleaned data to a new CSV file
    cleaned_df = pd.DataFrame(valid_smiles, columns=["Smiles"])
    cleaned_df.to_csv("Cleaned_Smiles.csv", index=False)

    print(f"Original: {len(df)} SMILES | Cleaned: {len(cleaned_df)} SMILES")
    return cleaned_df

# Example usage
file_path = "Smiles.csv"  # Make sure to use the correct path to your file
cleaned_df = load_and_clean_data(file_path)



Validating SMILES: 100%|██████████| 249455/249455 [01:07<00:00, 3673.17it/s]


Original: 249455 SMILES | Cleaned: 249455 SMILES


Check for valid smiles and converting to SELFIES

In [ ]:
import selfies as sf
from rdkit import Chem
from tqdm import tqdm
import pandas as pd

def is_valid_smiles(smi):
    """Check if a SMILES string is valid using RDKit."""
    return Chem.MolFromSmiles(smi) is not None

def preprocess_smiles(df):
    """Validate and clean SMILES before encoding."""
    df["smiles"] = df["smiles"].str.strip()  # Remove newline characters and spaces
    valid_smiles = [smi for smi in df["smiles"] if is_valid_smiles(smi)]
    print(f"Valid SMILES: {len(valid_smiles)} | Invalid Removed: {len(df) - len(valid_smiles)}")
    return pd.DataFrame(valid_smiles, columns=["smiles"])

def smiles_to_selfies(df):
    """Convert SMILES to SELFIES after validation."""
    selfies_list, invalid_smiles = [], []

    for smi in tqdm(df["smiles"], desc="Converting to SELFIES"):
        try:
            selfie = sf.encoder(smi)
            selfies_list.append(selfie)
        except sf.EncoderError as e:
            print(f"Failed SMILES: {smi} | Error: {e}")
            invalid_smiles.append(smi)
            selfies_list.append(None)

    df["SELFIES"] = selfies_list
    df = df.dropna(subset=["SELFIES"])
    df.to_csv("Selfies_Encoded.csv", index=False)

    print(f"Converted {len(df)} molecules to SELFIES. Skipped {len(invalid_smiles)} invalid SMILES.")
    return df

# Load dataset
file_path = "/content/drive/MyDrive/Smiles_data/250k_rndm_zinc_drugs_clean_3.csv"
df = pd.read_csv(file_path)

# Preprocess and validate SMILES
df_valid = preprocess_smiles(df)

# Convert valid SMILES to SELFIES
df_selfies = smiles_to_selfies(df_valid)


Valid SMILES: 249455 | Invalid Removed: 0


Converting to SELFIES: 100%|██████████| 249455/249455 [01:30<00:00, 2760.93it/s]


Converted 249455 molecules to SELFIES. Skipped 0 invalid SMILES.


 **Head**

In [ ]:
import pandas as pd

# Load the encoded SELFIES file
df_selfies = pd.read_csv("Selfies_Encoded.csv")

# Display the first few rows
print(df_selfies.head())


                                              smiles  \
0            CC(C)(C)c1ccc2occ(CC(=O)Nc3ccccc3F)c2c1   
1       C[C@@H]1CC(Nc2cncc(-c3nncn3C)c2)C[C@@H](C)C1   
2  N#Cc1ccc(-c2ccc(O[C@@H](C(=O)N3CCCC3)c3ccccc3)...   
3  CCOC(=O)[C@@H]1CCCN(C(=O)c2nc(-c3ccc(C)cc3)n3c...   
4  N#CC1=C(SCC(=O)Nc2cccc(Cl)c2)N=C([O-])[C@H](C#...   

                                             SELFIES  
0  [C][C][Branch1][C][C][Branch1][C][C][C][=C][C]...  
1  [C][C@@H1][C][C][Branch2][Ring1][Ring2][N][C][...  
2  [N][#C][C][=C][C][=C][Branch2][Ring2][Ring2][C...  
3  [C][C][O][C][=Branch1][C][=O][C@@H1][C][C][C][...  
4  [N][#C][C][=C][Branch2][Ring1][Ring1][S][C][C]...  


Load the SELFIES Dataset

Create a Vocabulary from SELFIES Characters

Create Token-to-ID and ID-to-Token Mappings

Tokenization of SELFIES

Apply Random Masking (15% Probability)

Save Tokenized and Masked Data

In [ ]:
import pandas as pd
import random
from transformers import BertTokenizerFast

# Load the SELFIES dataset
df_selfies = pd.read_csv("Selfies_Encoded.csv")

# Create a vocabulary from SELFIES characters
unique_tokens = set()
for selfies in df_selfies["SELFIES"]:
    unique_tokens.update(list(selfies))

# Add special tokens
unique_tokens.update(["[PAD]", "[CLS]", "[SEP]", "[MASK]"])

# Create a mapping from tokens to indices
token2id = {token: idx for idx, token in enumerate(sorted(unique_tokens))}
id2token = {idx: token for token, idx in token2id.items()}

# Tokenization function
def tokenize_selfies(selfies):
    return [token2id[token] for token in selfies]

# Apply tokenization
df_selfies["Tokenized"] = df_selfies["SELFIES"].apply(tokenize_selfies)

# Masking function
def apply_masking(tokenized_seq, mask_prob=0.15):
    masked_seq = []
    for token in tokenized_seq:
        if random.random() < mask_prob:
            masked_seq.append(token2id["[MASK]"])
        else:
            masked_seq.append(token)
    return masked_seq

# Apply masking
df_selfies["Masked"] = df_selfies["Tokenized"].apply(apply_masking)

# Save processed data
df_selfies.to_csv("Tokenized_Masked_SELFIES.csv", index=False)

print("Tokenization and masking completed. Processed data saved.")


Tokenization and masking completed. Processed data saved.


Load the Encoded SELFIES Dataset

Build Vocabulary from SELFIES Characters

Add Special Tokens ([PAD], [CLS], [SEP], [MASK])

Create Token-to-ID and ID-to-Token Mappings

Save the Vocabulary as JSON Files

Convert SELFIES to Token IDs with [CLS] and [SEP] Tokens

Apply Padding to Ensure Uniform Sequence Length

Generate Attention Masks for Padded Sequences

Save Tokenized Data to CSV

In [ ]:
import pandas as pd
import selfies as sf
import numpy as np
import json
from tqdm import tqdm
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Load the encoded SELFIES file
df_selfies = pd.read_csv("Selfies_Encoded.csv")

# Step 1: Build Vocabulary
unique_tokens = set()
for selfie in df_selfies["SELFIES"]:
    unique_tokens.update(sf.split_selfies(selfie))

# Add special tokens
special_tokens = ["[PAD]", "[CLS]", "[SEP]", "[MASK]"]
token_list = special_tokens + sorted(unique_tokens)

# Create Token-to-ID and ID-to-Token mappings
token2id = {token: idx for idx, token in enumerate(token_list)}
id2token = {idx: token for token, idx in token2id.items()}

# Save vocabulary
with open("token2id.json", "w") as f:
    json.dump(token2id, f)
with open("id2token.json", "w") as f:
    json.dump(id2token, f)

# Step 2: Convert SELFIES to Token IDs
def selfies_to_token_ids(selfie, token2id):
    tokens = ["[CLS]"] + list(sf.split_selfies(selfie)) + ["[SEP]"]  # Convert generator to list
    return [token2id[token] for token in tokens if token in token2id]

# Convert all SELFIES to Token IDs
max_len = 80  # Adjust based on dataset
tokenized_data = []

for selfie in tqdm(df_selfies["SELFIES"], desc="Converting to Token IDs"):
    token_ids = selfies_to_token_ids(selfie, token2id)
    padded = pad_sequences([token_ids], maxlen=max_len, padding="post", value=token2id["[PAD]"])[0]
    attention_mask = [1 if token != token2id["[PAD]"] else 0 for token in padded]

    tokenized_data.append([selfie, padded.tolist(), attention_mask])

# Save processed sequences to CSV
df_tokenized = pd.DataFrame(tokenized_data, columns=["SELFIES", "Token_Ids", "Attention_Mask"])
df_tokenized.to_csv("Tokenized_Data.csv", index=False)

print("Tokenization and sequence processing complete. Data saved to Tokenized_Data.csv.")


Converting to Token IDs: 100%|██████████| 249455/249455 [00:17<00:00, 14314.01it/s]


Tokenization and sequence processing complete. Data saved to Tokenized_Data.csv.


In [ ]:
!pip install datasets


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.4/491.4 kB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 17.2 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system 

In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"


Load Tokenized Data

Convert to Torch Tensors

Prepare MLM Labels

Split Data

Convert to Hugging Face Dataset

Load Pretrained BERT Model

Define Training Arguments

Initialize Trainer

Train Model

Save Trained Model

In [ ]:
import torch
import pandas as pd
import numpy as np
from transformers import BertTokenizerFast, BertForMaskedLM, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
from datasets import Dataset

# Load Tokenized Data
df = pd.read_csv("Tokenized_Data.csv")  # Update file path if needed
tokenized_sequences = df["Token_Ids"].apply(eval).tolist()  # Convert from string to list

# Convert to Torch Tensor with Attention Mask
max_len = max(len(seq) for seq in tokenized_sequences)  # Get max sequence length
input_ids = [seq + [0] * (max_len - len(seq)) for seq in tokenized_sequences]  # Padding
attention_mask = [[1] * len(seq) + [0] * (max_len - len(seq)) for seq in tokenized_sequences]  # Mask

input_ids = torch.tensor(input_ids)
attention_mask = torch.tensor(attention_mask)

# Labels are same as input_ids for MLM
labels = input_ids.clone()

# Split Data (80% Train, 10% Val, 10% Test)
train_ids, test_ids, train_mask, test_mask, train_labels, test_labels = train_test_split(
    input_ids, attention_mask, labels, test_size=0.2, random_state=42
)
val_ids, test_ids, val_mask, test_mask, val_labels, test_labels = train_test_split(
    test_ids, test_mask, test_labels, test_size=0.5, random_state=42
)

# Convert to Hugging Face Dataset
train_data = Dataset.from_dict({"input_ids": train_ids.tolist(), "attention_mask": train_mask.tolist(), "labels": train_labels.tolist()})
val_data = Dataset.from_dict({"input_ids": val_ids.tolist(), "attention_mask": val_mask.tolist(), "labels": val_labels.tolist()})

# Load BERT-Style Model
model = BertForMaskedLM.from_pretrained("bert-base-uncased")

# Define Optimized Training Arguments
training_args = TrainingArguments(
    output_dir="./bert_selfies_model",
    num_train_epochs=1,  # Reduce epochs
    per_device_train_batch_size=64,  # Larger batch size for speed
    save_steps=1000,  # Save less frequently
    save_total_limit=1,  # Keep only latest checkpoint
    logging_steps=500,  # Log every 500 steps
    fp16=True,  # Enable mixed precision
    report_to="none"  # Disable logging
)

# Define Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=val_data,
)

# Train Model
trainer.train()

# Save Model
model.save_pretrained("trained_bert_selfies")
print("Training Complete. Model Saved.")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Step,Training Loss
500,0.701100
1000,0.001800
1500,0.000600
2000,0.000400
2500,0.000300
3000,0.000300


Training Complete. Model Saved.


In [ ]:
test_data = Dataset.from_dict({
    "input_ids": test_ids.tolist(),
    "attention_mask": test_mask.tolist(),
    "labels": test_labels.tolist()
})


In [ ]:
results = trainer.evaluate(test_data)
print("Evaluation Results:", results)


Evaluation Results: {'eval_loss': 0.00016202354163397104, 'eval_runtime': 68.8475, 'eval_samples_per_second': 362.337, 'eval_steps_per_second': 45.303, 'epoch': 1.0}


loading of pretrained data on selfies

In [ ]:
from transformers import BertTokenizerFast, BertForMaskedLM
import torch

# Load tokenizer and model
tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")
model = BertForMaskedLM.from_pretrained("trained_bert_selfies")

# Set model to evaluation mode
model.eval()


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

BertForMaskedLM(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwi

In [ ]:
# Ensure special tokens are included
special_tokens = ["[PAD]", "[CLS]", "[SEP]", "[MASK]"]
unique_tokens.update(special_tokens)

# Recreate mappings
token2id = {token: idx for idx, token in enumerate(sorted(unique_tokens))}
id2token = {idx: token for token, idx in token2id.items()}

# Verify `[MASK]` exists
if "[MASK]" not in token2id:
    raise ValueError("Tokenizer does not contain `[MASK]` token!")
else:
    print('1')


1


In [ ]:
print("[MASK] ID:", token2id.get("[MASK]", "Not found"))


[MASK] ID: 65


In [ ]:
def tokenize_selfies(selfies):
    tokenized = []
    for token in selfies:
        if token in token2id:
            tokenized.append(token2id[token])
        else:
            print(f"Warning: Unknown token '{token}' found! Using [MASK] instead.")
            tokenized.append(token2id["[MASK]"])  # Use [MASK] for unknown tokens
    return tokenized


In [ ]:
def tokenize_selfies(selfies):
    return [token2id.get(token, token2id["[MASK]"]) for token in selfies]


In [ ]:
df["Tokenized"] = df["SELFIES"].apply(tokenize_selfies)


Convert SMILES to masked SELFIES

In [ ]:
import selfies as sf
import random

def smiles_to_masked_selfies(smiles, mask_token='[MASK]'):
    """Convert SMILES to SELFIES, mask one random token, and pass to generate_smiles."""

    try:
        # Convert SMILES to SELFIES
        selfies_seq = sf.encoder(smiles)
    except:
        print("Invalid SMILES input!")
        return None

    # Split SELFIES into tokens
    tokens = list(sf.split_selfies(selfies_seq))

    if not tokens:
        print("Could not tokenize SELFIES!")
        return None

    # Randomly select a position to mask
    mask_idx = random.randint(0, len(tokens) - 1)
    tokens[mask_idx] = mask_token

    # Reconstruct the masked SELFIES string
    masked_selfies = ''.join(tokens)

    print(f"Original SELFIES: {selfies_seq}")
    print(f"Masked SELFIES:   {masked_selfies}")

    # Pass to generate_smiles
    return generate_smiles(masked_selfies)


Tokenize Input SELFIES

Convert to Tensor Format

Perform Masked Token Prediction

Identify [MASK] Positions

Generate Top-k Predictions

Replace [MASK] with Predicted Tokens

Return Generated SELFIES

In [ ]:
def generate_smiles(masked_selfies, top_k=5):
    """ Generate new SELFIES by predicting masked tokens. """

    # Tokenize input sequence
    tokenized_seq = tokenize_selfies(masked_selfies)

    # Convert to tensor and reshape for model input
    input_tensor = torch.tensor([tokenized_seq])

    # Predict masked tokens
    with torch.no_grad():
        outputs = model(input_tensor)
        logits = outputs.logits  # Extract model outputs

    # Find `[MASK]` positions
    mask_token_id = token2id["[MASK]"]
    mask_positions = [i for i, token in enumerate(tokenized_seq) if token == mask_token_id]

    if not mask_positions:
        print(" No [MASK] token found in input!")
        return None

    print(f"Mask Positions: {mask_positions}")  # Debugging

    new_selfies = masked_selfies  # Copy input sequence

    # Iterate through each `[MASK]` position
    for idx in mask_positions:
        # Get probabilities and top-k predictions
        probs = torch.softmax(logits[0, idx], dim=-1)
        top_k_tokens = torch.topk(probs, top_k).indices.tolist()  # Ensure `top_k_tokens` is defined!

        for token in top_k_tokens:
            predicted_token = id2token.get(token, "[UNK]")  # Convert token ID to actual token

            #  Debug before replacing
            print(f"Replacing [MASK] at index {idx} with: {predicted_token}")

            #  Skip special tokens
            if predicted_token in ["[PAD]", "[UNK]", "[MASK]", "[CLS]", "[SEP]"]:
                continue  # Skip invalid tokens

            # Replace only one `[MASK]` at a time
            new_selfies = new_selfies.replace("[MASK]", predicted_token, 1)
            break  # Move to next `[MASK]` after replacing one

    print(f" Final Generated SELFIES: {new_selfies}")
    return new_selfies  # Return final result after all replacements




In [ ]:
# Call this function wit SMILES string
smiles_to_masked_selfies("CC(=O)OC")  # Example SMILES for acetic acid


Original SELFIES: [C][C][=Branch1][C][=O][O][C]
Masked SELFIES:   [C][C][=Branch1][C][=O][MASK][C]
Mask Positions: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31]
Replacing [MASK] at index 0 with: [MASK]
Replacing [MASK] at index 0 with: [=P]
Replacing [MASK] at index 1 with: [MASK]
Replacing [MASK] at index 1 with: [=P]
Replacing [MASK] at index 2 with: [MASK]
Replacing [MASK] at index 2 with: [=P]
Replacing [MASK] at index 3 with: [MASK]
Replacing [MASK] at index 3 with: [=P]
Replacing [MASK] at index 4 with: [MASK]
Replacing [MASK] at index 4 with: [=P]
Replacing [MASK] at index 5 with: [MASK]
Replacing [MASK] at index 5 with: [=P]
Replacing [MASK] at index 6 with: [MASK]
Replacing [MASK] at index 6 with: [=P]
Replacing [MASK] at index 7 with: [MASK]
Replacing [MASK] at index 7 with: [C@H1]
Replacing [MASK] at index 8 with: [MASK]
Replacing [MASK] at index 8 with: [C@H1]
Replacing [MASK] at index 9 with: [MASK]
R

'[C][C][=Branch1][C][=O][=P][C]'

In [ ]:
generate_smiles("[C][MASK][O][C][=O][O]")


Mask Positions: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21]
Replacing [MASK] at index 0 with: [MASK]
Replacing [MASK] at index 0 with: [=P]
Replacing [MASK] at index 1 with: [MASK]
Replacing [MASK] at index 1 with: [=P]
Replacing [MASK] at index 2 with: [MASK]
Replacing [MASK] at index 2 with: [=P]
Replacing [MASK] at index 3 with: [MASK]
Replacing [MASK] at index 3 with: [=P]
Replacing [MASK] at index 4 with: [MASK]
Replacing [MASK] at index 4 with: [=P]
Replacing [MASK] at index 5 with: [MASK]
Replacing [MASK] at index 5 with: [C@H1]
Replacing [MASK] at index 6 with: [MASK]
Replacing [MASK] at index 6 with: [C@H1]
Replacing [MASK] at index 7 with: [MASK]
Replacing [MASK] at index 7 with: [C@H1]
Replacing [MASK] at index 8 with: [MASK]
Replacing [MASK] at index 8 with: [C@H1]
Replacing [MASK] at index 9 with: [MASK]
Replacing [MASK] at index 9 with: [C@H1]
Replacing [MASK] at index 10 with: [MASK]
Replacing [MASK] at index 10 with: [C@H1]
Replacing [

'[C][=P][O][C][=O][O]'

In [ ]:
import selfies as sf

def convert_selfies_to_smiles(new_selfies):
    """
    Converts a SELFIES string to a SMILES string using selfies.decoder.
    """
    try:
        smiles = sf.decoder(new_selfies)
        print("SELFIES:", new_selfies)
        print("SMILES: ", smiles)
        return smiles
    except Exception as e:
        print("Decoding failed:", e)
        return None





Checking for new SMILES

In [ ]:
import pandas as pd

# Load the dataset
df_selfies = pd.read_csv("Selfies_Encoded.csv")

# Check if the SMILES is in the dataset
smiles_to_check = "CC(=O)N"

if smiles_to_check in df_selfies["SELFIES"].values:
    print("SMILES is present in the dataset.")
else:
    print("SMILES is NOT present in the dataset.")


SMILES is NOT present in the dataset.


Validity check

In [ ]:
from rdkit import Chem

# Generated SMILES
smiles = "CC(=O)N"

# Check if the SMILES is valid
mol = Chem.MolFromSmiles(smiles)

if mol:
    print("Valid SMILES!")
else:
    print("Invalid SMILES!")


Valid SMILES!


Compile all functions like:
 Masking the sekfies
 Generating new SELFIES
 SELFIES to SMILES

In [ ]:
# Assuming these are already defined: generate_smiles, convert_selfies_to_smiles

# Get a masked SELFIES using an input SMILES
masked_selfies = smiles_to_masked_selfies("CC(=O)N")  # Acetic acid

# Generate new SELFIES by replacing [MASK]
if masked_selfies:
    new_selfies = generate_smiles(masked_selfies)

    # Decode the new SELFIES to SMILES
    if new_selfies:
        convert_selfies_to_smiles(new_selfies)


Original SELFIES: [C][C][=Branch1][C][=O][N]
Masked SELFIES:   [MASK][C][=Branch1][C][=O][N]
Mask Positions: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28]
Replacing [MASK] at index 0 with: [MASK]
Replacing [MASK] at index 0 with: [=P]
Replacing [MASK] at index 1 with: [MASK]
Replacing [MASK] at index 1 with: [=P]
Replacing [MASK] at index 2 with: [MASK]
Replacing [MASK] at index 2 with: [C@H1]
Replacing [MASK] at index 3 with: [MASK]
Replacing [MASK] at index 3 with: [C@H1]
Replacing [MASK] at index 4 with: [MASK]
Replacing [MASK] at index 4 with: [C@H1]
Replacing [MASK] at index 5 with: [MASK]
Replacing [MASK] at index 5 with: [C@H1]
Replacing [MASK] at index 6 with: [MASK]
Replacing [MASK] at index 6 with: [C@H1]
Replacing [MASK] at index 7 with: [MASK]
Replacing [MASK] at index 7 with: [C@H1]
Replacing [MASK] at index 8 with: [MASK]
Replacing [MASK] at index 8 with: [C@H1]
Replacing [MASK] at index 9 with: [MASK]
Replacing

Compile in one function

In [ ]:
def full_selfies_to_smiles_pipeline(smiles_input):
    masked_selfies = smiles_to_masked_selfies(smiles_input)
    if not masked_selfies:
        return None

    new_selfies = generate_smiles(masked_selfies)
    if not new_selfies:
        return None

    new_smiles = convert_selfies_to_smiles(new_selfies)
    return new_smiles

# Example usage:
result = full_selfies_to_smiles_pipeline("CC(=O)N")


Original SELFIES: [C][C][=Branch1][C][=O][N]
Masked SELFIES:   [MASK][C][=Branch1][C][=O][N]
Mask Positions: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28]
Replacing [MASK] at index 0 with: [MASK]
Replacing [MASK] at index 0 with: [=P]
Replacing [MASK] at index 1 with: [MASK]
Replacing [MASK] at index 1 with: [=P]
Replacing [MASK] at index 2 with: [MASK]
Replacing [MASK] at index 2 with: [C@H1]
Replacing [MASK] at index 3 with: [MASK]
Replacing [MASK] at index 3 with: [C@H1]
Replacing [MASK] at index 4 with: [MASK]
Replacing [MASK] at index 4 with: [C@H1]
Replacing [MASK] at index 5 with: [MASK]
Replacing [MASK] at index 5 with: [C@H1]
Replacing [MASK] at index 6 with: [MASK]
Replacing [MASK] at index 6 with: [C@H1]
Replacing [MASK] at index 7 with: [MASK]
Replacing [MASK] at index 7 with: [C@H1]
Replacing [MASK] at index 8 with: [MASK]
Replacing [MASK] at index 8 with: [C@H1]
Replacing [MASK] at index 9 with: [MASK]
Replacing